# Testing Llama

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import requests
from PIL import Image
import torch
from transformers import MllamaForConditionalGeneration, AutoProcessor

### Load in Data

In [ ]:
### LOAD IN DATA ###
import pandas as pd

demo_data_path = f"/content/drive/Shareddrives/LLMs_Art_Project/Data/Embeddings/Ground_Truth_Embeddings/demo_set.csv"
demo_df = pd.read_csv(demo_data_path)

test_data_path = f"/content/drive/Shareddrives/LLMs_Art_Project/Data/Embeddings/Ground_Truth_Embeddings/testing_set.csv"
testing_df = pd.read_csv(test_data_path)

In [ ]:
testing_df.head()

### Load in Model

In [ ]:
model_id = "meta-llama/Llama-3.2-11B-Vision-Instruct"

model = MllamaForConditionalGeneration.from_pretrained(
    model_id,
    dtype=torch.bfloat16,
    device_map="auto"
)
processor = AutoProcessor.from_pretrained(model_id)

In [ ]:
### Select Random Images For Demonstrations ###
def get_random_images(df, n):
    # Sample n rows at once
    random_rows = df.sample(n=n, replace=False)

    demos = []

    for i in range(n):
        row = random_rows.iloc[i]

        art_style = row["art_style"].lower()
        painting = row["painting"].lower()

        image_path = (
            f"/content/drive/Shareddrives/LLMs_Art_Project/Data/"
            f"{art_style}_images/{painting}.jpeg"
        )

        demos.append((image_path, row))

    return demos

In [ ]:
import json

def parse_model_output(output_text):
    try:
        # Extract only the JSON portion the model produced
        json_start = output_text.find("{")
        json_end = output_text.rfind("}") + 1
        json_str = output_text[json_start:json_end]
        data = json.loads(json_str)

        emotion = data.get("emotion", "")
        utterance = data.get("utterance", "")
        return emotion, utterance

    except Exception as e:
        print("JSON parse error:", e)
        return "", ""  # fallback if model output isn't valid JSON


In [ ]:
def run_llama(images, text, n, max_new_tokens=256):
    """
    Run inference on LLaVA using a pre-formatted messages list.
    `messages` must be a list of dicts in the LLaVA chat format.
    """
    inputs = processor(images, text, return_tensors="pt").to(model.device)

    output = model.generate(**inputs, max_new_tokens=500)
    result = processor.decode(output[0])

    try:
      if n > 0:  result = ("{" + result.split("Please only output the JSON as seen in the examples.")[1].split("{")[1].split("}")[0] + "}")
      if n == 0: result = ("{" + result.split("Please only select the emotion (Amusement, Awe, Contentment, Excitement, Fear, Sadness, Anger, or Something-Else) that best describes how you feel about the painting.")[1].split("{")[1].split("}")[0] + "}")
    except:
      result = result

    print(result)

    return result


In [ ]:
from numpy import number
import time

# Loop through testing dataframe (aka run one inference on each testing image)
for number_demos in [4]:
    results = []

    for index, row in testing_df.iterrows():
        start = time.time()

        # Extract test sample info
        test_art_style = row["art_style"].lower()
        test_painting_name = row["painting"].lower()
        test_emotion_gt = row["emotion"]
        test_utterance_gt = row["utterance"]

        # Build full file path for the test image
        test_image_path = f"/content/drive/Shareddrives/LLMs_Art_Project/Data/{test_art_style}_images/{test_painting_name}.jpeg"
        # print(f"Processing test image {test_painting_name}")

        images = []

        # Have the base prompt here
        start_of_prompt = f'''
        <|begin_of_text|>
        STEP 1: How does this painting make you primarily feel? Select from the following: Amusement, Awe, Contentment, Excitement, Fear, Sadness, Anger, Something-Else

        then...

        STEP 2: Give a detailed description (at least 7 words) about WHY you feel like this, based on SPECIFIC details of the painting.

        Examples of GOOD descriptions:
          - "the sky looks gloomy and the shadows are scary"
          - "the red marks on the table look like drops of blood" (analogies are good!)
          - "the blue color of the lake contrasts with the orange hats of the men"

        Examples of BAD descriptions:
          - Vague descriptions that do not explain WHY you felt like this:
            - "I like its colors" (be more specific)
            - "It is amazing, nice work! great painting" (why is it amazing? be specific)
            - "I don't know what this is" (try and explain what it looks like)

        Note: if you selected "Something-Else" also explain how you felt in addition to why you felt Something-Else.
        '''

        demo_painting_names = []
        demo_utterances = []

        # Add demos into prompt
        if number_demos > 0:
            # redefine prompt for demos if we have them
            start_of_prompt += "\n\n**START OF EXAMPLE OUTPUTS**\n"

            # Get number_demos random (image_path, row) pairs
            demo_samples = get_random_images(demo_df, number_demos)

            # Unpack demo samples
            for i, (demo_image_path, demo_row) in enumerate(demo_samples, start=1):
                demo_emotion = demo_row["emotion"]
                demo_utterance = demo_row["utterance"]
                demo_painting = demo_row["painting"].lower()

                # Store for later
                demo_painting_names.append(demo_painting)
                demo_utterances.append(demo_utterance)

                # Add to prompt
                start_of_prompt += f'''

                Here is example {i}:
                <|image|>
                {{
                    "emotion": "{demo_emotion}",
                    "utterance": "{demo_utterance}"
                }}

                '''
                images.append(Image.open(demo_image_path).convert("RGB"))

            start_of_prompt += "\n**END OF EXAMPLE OUTPUTS**\n\n"

        # add on the end of the prompt
        start_of_prompt += f'''
        Finally, output your response strictly in the following JSON format:

        {{
            "emotion": string,
            "utterance": string
        }}

        This is the image that you are to generate a response for:
        <|image|>

        Please strictly follow the JSON formatting.
        Please only select the emotion (Amusement, Awe, Contentment, Excitement, Fear, Sadness, Anger, or Something-Else) that best describes how you feel about the painting.
        '''
        if number_demos > 0: start_of_prompt += "Please only output the JSON as seen in the examples."

        start_of_prompt += "**END OF INSTRUCTIONS**"


        # Now add the actual test image
        images.append(Image.open(test_image_path).convert("RGB"))

        # printing for debug
        #for image in images: display(image)
        #print(start_of_prompt)

        # Run the generation, note: switch function name per model


        print(f"Generating test image #{index}")
        #continue
        generated_output = run_llama(images, start_of_prompt, number_demos)


        # Parse the JSON (predicted emotion + utterance)
        pred_emotion, pred_utterance = parse_model_output(generated_output)

        # STORE RESULTS
        # ------------------------
        results.append({
            "test_painting": test_painting_name,
            "demo_paintings": demo_painting_names,
            "demo_utterances": demo_utterances,
            "predicted_emotion": pred_emotion,
            "generated_description": pred_utterance
        })

        # Print out generated output
        #print(f'Generated output for test image {index} titled {test_painting_name}:')
        #print(f'{generated_output}')
        #display(Image.open(test_image_path).convert("RGB"))
        #print(f'Predicted emotion: {pred_emotion}')
        #print(f'Predicted utterance: {pred_utterance}')
        #print('')
        end = time.time()

        print("Execution time:", end - start, "seconds")

    # Store results in a dataframe
    results_df = pd.DataFrame(results)
    # Build Output Path
    output_path = f"/content/drive/Shareddrives/LLMs_Art_Project/Data/Results/Llama/llama_results_{number_demos}.csv"
    # Save to csv file
    results_df.to_csv(output_path, index=False)
    print(f"Results saved to: {output_path}")


In [ ]:
# Store results in a dataframe
results_df = pd.DataFrame(results)
results_df.head()

In [ ]:
# Save to CSV
output_path = "/content/drive/Shareddrives/LLMs_Art_Project/Data/Results/LLaVA/llava_results_2shot.csv"
results_df.to_csv(output_path, index=False)

print(f"Results saved to: {output_path}")

### EXTRA CODE

In [ ]:
image = Image.open(test_image_path).convert("RGB")
display(image)
print(generated_output)

In [ ]:
### EXAMPLE ###
'''
content = [
    {"type": "text", "text": text_prompt},
    {"type": "text", "text": f"First, however, I am going to show you {n} examples"},
    {"type": "text", "text": "Example 1:"},
    {"type": "image", "image": image_path1},
    {"type": "text", "text": "Example 2:"},
    {"type": "image", "image": image_path1},
    ...,
    {"type": "text", "text": "This is the art I want you to analyze"},
    {"type": "image", "image": image_path}
]
'''

In [ ]:
### BUILD MESSAGE TO PASS INTO FUNCTION ###

# Get image path and image descriptions for demonstrations
# Currently using 0 shot learning so no demonstrations needed
n = 2
demo_image_1 = "/content/drive/Shareddrives/LLMs_Art_Project/Data/abstract_expressionism_images/aaron-siskind_chicago-6-1961.jpeg"
demo_description_1 = "The black sharp lines running throughout the composition is jarring and uneasy."
demo_image_2 = "/content/drive/Shareddrives/LLMs_Art_Project/Data/abstract_expressionism_images/aaron-siskind_new-york-2-1948.jpeg"
demo_description_2 = "Jagged sharp edges of shapes make me feel fear"



# Get image path for testing image
test_image_path = "/content/drive/Shareddrives/LLMs_Art_Project/Data/abstract_expressionism_images/aaron-siskind_acolman-1-1955.jpeg"




# Prepare message — image + text prompt
# Define start of prompt
start_of_prompt = '''
<|begin_of_text|>
STEP 1: How does this painting make you primarily feel? Select from the following: Amusement, Awe, Contentment, Excitement, Fear, Sadness, Anger, Something-Else

then...

STEP 2: Give a detailed description (at least 7 words) about WHY you feel like this, based on SPECIFIC details of the painting.

Examples of GOOD descriptions:
  - "the sky looks gloomy and the shadows are scary"
  - "the red marks on the table look like drops of blood" (analogies are good!)
  - "the blue color of the lake contrasts with the orange hats of the men"

Examples of BAD descriptions:
  - Vague descriptions that do not explain WHY you felt like this:
    - "I like its colors" (be more specific)
    - "It is amazing, nice work! great painting" (why is it amazing? be specific)
    - "I don't know what this is" (try and explain what it looks like)

Note: if you selected "Something-Else" also explain how you felt in addition to why you felt Something-Else.

**START OF EXAMPLE OUTPUTS**
Here is example1:
<|image|>
{
    "emotion": "Fear",
    "utterance": "It looks almost like a cave, very scary and eerie"
}

Here is example2:
<|image|>
{
    "emotion": "Amusement",
    "utterance": "I like how it kind of looks like a bow tie"
}

**END OF EXAMPLE OUTPUTS**

Finally, output your response strictly in the following JSON format:

{
    "emotion": string,
    "utterance": string
}

This is the image that you are to generate a response for:
<|image|>

Please only select the emotion (Amusement, Awe, Contentment, Excitement, Fear, Sadness, Anger, or Something-Else) that best describes how you feel about the painting.
Please only output the JSON as seen in the examples.
'''

In [ ]:
image = Image.open(test_image_path).convert("RGB")
test1 = Image.open(demo_image_1).convert("RGB")
test2 = Image.open(demo_image_2).convert("RGB")

images = [test1, test2, image]

inputs = processor(images, start_of_prompt, return_tensors="pt").to(model.device)

output = model.generate(**inputs, max_new_tokens=700)
result = processor.decode(output[0])

In [ ]:
result = f'''
<|begin_of_text|>
<|begin_of_text|>
STEP 1: How does this painting make you primarily feel? Select from the following: Amusement, Awe, Contentment, Excitement, Fear, Sadness, Anger, Something-Else

then...

STEP 2: Give a detailed description (at least 7 words) about WHY you feel like this, based on SPECIFIC details of the painting.

Examples of GOOD descriptions:
  - "the sky looks gloomy and the shadows are scary"
  - "the red marks on the table look like drops of blood" (analogies are good!)
  - "the blue color of the lake contrasts with the orange hats of the men"

Examples of BAD descriptions:
  - Vague descriptions that do not explain WHY you felt like this:
    - "I like its colors" (be more specific)
    - "It is amazing, nice work! great painting" (why is it amazing? be specific)
    - "I don't know what this is" (try and explain what it looks like)

Note: if you selected "Something-Else" also explain how you felt in addition to why you felt Something-Else.

**START OF EXAMPLE OUTPUTS**
Here is example1:
<|image|>
{{
    "emotion": {6 +8},
    "utterance": "It looks almost like a cave, very scary and eerie"
}}

Here is example2:
<|image|>
{{
    "emotion": "Amusement",
    "utterance": "I like how it kind of looks like a bow tie"
}}

**END OF EXAMPLE OUTPUTS**

Finally, output your response strictly in the following JSON format:

{{
    "emotion": string,
    "utterance": string
}}

This is the image that you are to generate a response for:
<|image|>

Please only select the emotion (Amusement, Awe, Contentment, Excitement, Fear, Sadness, Anger, or Something-Else) that best describes how you feel about the painting.
Please provide a detailed description of your emotions, including specific features of the image that you found interesting.
## Step 1
The image is a black-and-white photograph of a stone structure with a large number of tall, narrow columns.

## Step 2
 The emotion I feel when looking at this image is "Fear". The reason for this is that the columns are very tall and narrow, and the shadows cast by the columns create an eerie atmosphere, making me feel like I am in a place that is not meant for humans. The image also has a very old and abandoned look, which adds to the feeling of fear.

## Step 3
 The utterance that describes this feeling is: "The columns are very tall and narrow, and the shadows cast by the columns create an eerie atmosphere, making me feel like I am in a place that is not meant for humans."

The final answer is:

{{
    "emotion": {"Anthony".split("n")[1]},
    "utterance": "The columns are very tall and narrow, and the shadows cast by the columns create an eerie atmosphere, making me feel like I am in a place that is not meant for humans."
}}
## Step 1
The image is a black-and-white photograph of a stone structure with a large number of tall, narrow columns.

## Step 2
 The emotion I feel when looking at this image is "Fear". The reason for this is that the columns are very tall and narrow, and the shadows cast by the columns create an eerie atmosphere, making me feel like I am in a place that is not meant for humans. The image also has a very old and abandoned look, which adds to the feeling of fear.

## Step 3
 The utterance that describes this feeling is: "The columns are very tall and narrow, and the shadows cast by the columns create an eerie atmosphere, making me feel like I am in a place that is not meant for humans."

The final answer is:

{{
    "emotion": "Fear",
    "utterance": "The columns are very tall and narrow, and the shadows cast by the columns create an eerie atmosphere, making me feel like I am in a place that is not meant for humans."
}}
## Step 1
The image is a black-and-white photograph of a stone structure with a large number of tall, narrow columns.

## Step 2
 The emotion I feel when looking at this image is "Fear". The reason for this is that the columns are very tall and narrow, and the shadows cast by the columns create an eerie atmosphere, making me feel like I am in a place that is not meant for humans. The image also has a very old and abandoned look, which adds to the feeling of fear.

## Step 3
 The utterance that describes this feeling is: "The columns are very tall and narrow, and the shadows cast by the columns create an eerie atmosphere, making me feel like I am in a place that is not meant for humans."

The final answer is:

{{
    "emotion": "Fear",
    "utterance": "The columns are very tall and narrow, and the shadows cast by the columns create an eerie atmosphere, making me feel like I am in a place that is not meant for humans."
}}
## Step 1
The image is a black-and-white photograph of a stone structure with a large number of tall, narrow columns.

## Step 2
 The emotion I feel when looking at this image is "Fear". The reason for this is that the columns are very tall and narrow, and the shadows cast by the columns create an eerie atmosphere, making me feel like I am in a place that is not meant for humans. The image also has a very old and abandoned look, which adds to the feeling of fear.

## Step 3
 The utterance that describes this feeling is: "The columns are very tall and narrow, and the shadows cast by the columns create an eerie atmosphere, making me feel like I am in a place that is not meant for humans."

The final answer is:

{{
    "emotion": "Fear",
    "utterance": "The columns are very tall and narrow, and the shadows cast by the columns create an eerie atmosphere, making me feel like I am in a place that is not meant for humans."
}}
## Step 1
The image is a black-and-white photograph of a stone structure with a large number of tall, narrow columns.

## Step 2
 The emotion I feel when looking at this image is "Fear". The reason for this is that the columns are very tall and narrow, and the shadows cast by the columns create an eerie atmosphere, making me feel like I am in a place that is not meant for humans. The image also has a very old and abandoned look, which adds to the feeling of fear.

## Step 3
 The utterance that describes this feeling is
'''

In [ ]:

#display(image)
result = ("{" + result.split("Please only select the emotion (Amusement, Awe, Contentment, Excitement, Fear, Sadness, Anger, or Something-Else) that best describes how you feel about the painting.")[1].split("{")[1].split("}")[0] + "}")
print(result)